# CMSC 678 Project 

## 0. User configuration

Edit `HOME_FOLDER` to point to where the BCI data should live on Google Drive.

> ⚠️ Use **plain spaces** (not `\ ` escapes) — Python handles spaces in paths fine, but `\ ` becomes a literal backslash that breaks `os.path.join`.

In [10]:
!nvidia-smi

Thu May  7 12:15:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             50W /  400W |    1050MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [11]:
# === USER CONFIG ===========================================================
HOME_FOLDER = '/content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project'
DATA_DIR    = HOME_FOLDER + '/data'   # all .npz + A0xE.mat will end up here
# ===========================================================================

## 1. Mount Google Drive

In [12]:
import os

# Mount Drive once; skip if already mounted (re-running this cell is safe).
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DATA_DIR, exist_ok=True)
print(f'HOME_FOLDER: {HOME_FOLDER}\nDATA_DIR:    {DATA_DIR}')

HOME_FOLDER: /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project
DATA_DIR:    /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project/data


In [13]:
!ls /content/drive/MyDrive/Masters/'First Year'/'CMSC 678'/CMSC678Project

ablation.py	     file_dependency_diagram.png  reimplementation
ATCNet.py	     MAIN_RUN.py		  requirements.txt
data		     Mamba.py			  results
desc_2a.pdf	     normal_results.py		  sr_augmentation.py
EEGNetBYOL.py	     paper.pdf			  summary.txt
EEGNet.py	     plot.py			  Untitled0.ipynb
FBCSP_Multiclass.py  __pycache__		  utils.py
FBCSP_V4.py	     README.txt


## 2. Install dependencies

`mamba-ssm` is installed separately because it requires `--no-build-isolation`.

In [14]:
%cd "$HOME_FOLDER"

#! pip install -r requirements.txt --force-reinstall
!pip install -q -r requirements.txt
!pip install -q mamba-ssm --no-build-isolation

/content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project


## 3. Download data

Pulls everything we need into `DATA_DIR` in a single pass:

- **18 `.npz` signal files** (~600 MB total) from [`bregydoc/bcidatasetIV2a`](https://github.com/bregydoc/bcidatasetIV2a) — already in the `s` / `epos` / `etyp` / `edur` / `artifacts` format that `utils.load_data` expects, no conversion needed.
- **9 `A0xE.mat` label files** (~few KB total) extracted from `true_labels.zip` on [bbci.de](http://bbci.de/competition/iv/results/ds2a/) — the eval-session class labels that were withheld during the original competition.

**Safeguards against re-downloading:**
- `.npz` files are skipped if present and > 25 MB.
- Downloads write to `*.partial` first and rename on success → if a download crashes mid-stream, the next run retries cleanly without leaving a corrupt half-file in place.
- The `true_labels.zip` fetch + extract is skipped entirely if all 9 `A0xE.mat` files already exist.
- Already-extracted zip members are not re-extracted.

In [15]:
import urllib.request, zipfile

NPZ_BASE   = 'https://raw.githubusercontent.com/bregydoc/bcidatasetIV2a/master/'
LABELS_URL = 'http://bbci.de/competition/iv/results/ds2a/true_labels.zip'

# Atomic download: write to *.partial then rename. A crash mid-download leaves
# the .partial behind instead of a corrupt final file, so the next run retries.
def download_atomic(url, dest):
    tmp = dest + '.partial'
    urllib.request.urlretrieve(url, tmp)
    os.replace(tmp, dest)

# ---- 1. Signal .npz files (skip any > 25 MB; that means it's complete) ----
print('Signals (.npz):')
for i in range(1, 10):
    for suffix in ('T', 'E'):
        fname = f'A{i:02d}{suffix}.npz'
        dest  = os.path.join(DATA_DIR, fname)
        if os.path.exists(dest) and os.path.getsize(dest) > 25_000_000:
            print(f'  {fname:>10}  cached     ({os.path.getsize(dest)/1e6:.1f} MB)')
            continue
        download_atomic(NPZ_BASE + fname, dest)
        print(f'  {fname:>10}  downloaded ({os.path.getsize(dest)/1e6:.1f} MB)')

# ---- 2. True eval labels (skip the whole zip step if all 9 .mat already exist) ----
print('\nLabels (A0xE.mat):')
need_labels = any(not os.path.exists(os.path.join(DATA_DIR, f'A{i:02d}E.mat')) for i in range(1, 10))

if not need_labels:
    print('  all 9 A0xE.mat files already present — skipping label fetch')
else:
    zip_path = os.path.join(DATA_DIR, 'true_labels.zip')
    if not os.path.exists(zip_path):
        download_atomic(LABELS_URL, zip_path)

    # Extract only members that aren't already on disk.
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.namelist():
            if not os.path.exists(os.path.join(DATA_DIR, member)):
                zf.extract(member, DATA_DIR)

    # Some BBCI releases name files A01.mat instead of A01E.mat — normalise.
    for i in range(1, 10):
        target   = os.path.join(DATA_DIR, f'A{i:02d}E.mat')
        fallback = os.path.join(DATA_DIR, f'A{i:02d}.mat')
        if not os.path.exists(target) and os.path.exists(fallback):
            os.rename(fallback, target)

# ---- 3. Verify all 27 files (18 .npz + 9 label .mat) ----
missing = [f for i in range(1,10)
             for f in (f'A{i:02d}T.npz', f'A{i:02d}E.npz', f'A{i:02d}E.mat')
             if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(f'Missing after download: {missing}')
print(f'\n✅ All 27 data files present in {DATA_DIR}')

Signals (.npz):
    A01T.npz  cached     (33.3 MB)
    A01E.npz  cached     (34.0 MB)
    A02T.npz  cached     (33.7 MB)
    A02E.npz  cached     (34.3 MB)
    A03T.npz  cached     (34.0 MB)
    A03E.npz  cached     (32.6 MB)
    A04T.npz  cached     (29.2 MB)
    A04E.npz  cached     (33.1 MB)
    A05T.npz  cached     (33.4 MB)
    A05E.npz  cached     (34.2 MB)
    A06T.npz  cached     (34.5 MB)
    A06E.npz  cached     (33.6 MB)
    A07T.npz  cached     (33.8 MB)
    A07E.npz  cached     (33.3 MB)
    A08T.npz  cached     (34.8 MB)
    A08E.npz  cached     (35.7 MB)
    A09T.npz  cached     (34.7 MB)
    A09E.npz  cached     (34.8 MB)

Labels (A0xE.mat):
  all 9 A0xE.mat files already present — skipping label fetch

✅ All 27 data files present in /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project/data


## 4. Sanity check

`utils.load_data()` was patched (see `reimplementation/changes.txt`) to read the data directory from the `BCI_DATA_DIR` environment variable, so we can point it at our `DATA_DIR` without forking the source tree. This cell loads subject 01 through the real `load_data` + `subject_to_arrays` pipeline and asserts the shapes. Expected: `(288, 22, 512)` after the 250 → 128 Hz polyphase resample.

In [16]:
import numpy as np, sys

# Set env var BEFORE importing project modules — normal_results.py reads it
# at import time to pick the data dir without us having to fork the source.
os.environ['BCI_DATA_DIR'] = DATA_DIR
if HOME_FOLDER not in sys.path:
    sys.path.insert(0, HOME_FOLDER)

from utils import load_data, subject_to_arrays

# Pass data_dir explicitly — env-var fallback is a safety net, not the primary path.
subjectData, subjectDataEVAL = load_data(data_dir=DATA_DIR)

# Run subject 01 through the full pipeline (250 Hz raw -> 128 Hz, per-channel scaled).
X_train, y_train, X_eval, y_eval = subject_to_arrays(
    subjectData['subject01'], subjectDataEVAL['subject01'],
    os.path.join(DATA_DIR, 'A01E.mat'))

print(f'X_train: {X_train.shape}   X_eval: {X_eval.shape}')
assert X_train.shape == X_eval.shape == (288, 22, 512)
assert set(np.unique(y_train)) == set(np.unique(y_eval)) == {0, 1, 2, 3}
print('✅ Data pipeline OK — ready for the experiment cells below.')

X_train: (288, 22, 512)   X_eval: (288, 22, 512)
✅ Data pipeline OK — ready for the experiment cells below.


---
## 5. Experiment 1 — FBCSP baseline

Runs `normal_results.FBCSP_results()` across all 9 subjects. See `understanding.txt § 6` for why this is experiment #1 (baseline · defines high/low split · pipeline sanity check).

Runtime: ~3 min CPU. Result is pickled to `RESULTS_DIR/fbcsp.pkl`. Set `FORCE_RERUN_FBCSP = True` in the cell below to ignore the cache and recompute.

In [17]:
import pickle, time

# Flip to True to ignore the pickle cache and re-run FBCSP from scratch.
FORCE_RERUN_FBCSP = False

RESULTS_DIR = os.path.join(HOME_FOLDER, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
fbcsp_cache = os.path.join(RESULTS_DIR, 'fbcsp.pkl')

# Cache schema: (accuracies, confusion_matrix, runtime_seconds).
# Backwards-compat: older pickles only had (acc, cm) — handle that case.
if os.path.exists(fbcsp_cache) and not FORCE_RERUN_FBCSP:
    with open(fbcsp_cache, 'rb') as f:
        loaded = pickle.load(f)
    fbcsp_acc, fbcsp_cm = loaded[0], loaded[1]
    fbcsp_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {fbcsp_runtime/60:.1f} min' if fbcsp_runtime else 'prior runtime not recorded'
    print(f'Loaded cached FBCSP results ({msg})')
else:
    print('Running FBCSP across all 9 subjects...')
    from normal_results import FBCSP_results
    t0 = time.perf_counter()
    fbcsp_acc, fbcsp_cm = FBCSP_results()
    fbcsp_runtime = time.perf_counter() - t0
    with open(fbcsp_cache, 'wb') as f:
        pickle.dump((fbcsp_acc, fbcsp_cm, fbcsp_runtime), f)
    print(f'Wall time: {fbcsp_runtime/60:.1f} min   cached -> {fbcsp_cache}')

fbcsp_acc = np.array(fbcsp_acc)

# Per-subject + summary stats.
print('\nPer-subject accuracy:')
for i, a in enumerate(fbcsp_acc, 1):
    print(f'  S{i:02d}: {a:.4f}')
print(f'  mean: {fbcsp_acc.mean():.4f}   median: {np.median(fbcsp_acc):.4f}   (paper mean: 0.634)')

# Median split — the foundation for the BCI inefficiency analysis (RQ1–RQ4).
median = np.median(fbcsp_acc)
high = [i for i, a in enumerate(fbcsp_acc, 1) if a >= median]
low  = [i for i, a in enumerate(fbcsp_acc, 1) if a <  median]
print(f'\nHigh (>= median): {high}    paper: [1, 3, 7, 8, 9]')
print(f'Low  (<  median): {low}    paper: [2, 4, 5, 6]')

# Aggregate confusion matrix — expect tongue/feet as the dominant confusion.
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], fbcsp_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached FBCSP results (prior runtime not recorded)

Per-subject accuracy:
  S01: 0.6944
  S02: 0.5347
  S03: 0.7986
  S04: 0.5868
  S05: 0.5208
  S06: 0.4062
  S07: 0.7847
  S08: 0.7118
  S09: 0.6944
  mean: 0.6370   median: 0.6944   (paper mean: 0.634)

High (>= median): [1, 3, 7, 8, 9]    paper: [1, 3, 7, 8, 9]
Low  (<  median): [2, 4, 5, 6]    paper: [2, 4, 5, 6]

Confusion matrix (rows=true, cols=pred):
              Left   Right  Feet   Tongue
  true Left       426    130     41     51
  true Right       82    471     66     29
  true Feet        91     62    367    128
  true Tongue      89     92     80    387


**Interpretation checklist** before moving on:
- mean accuracy is within ±0.02 of 0.634
- high/low split matches the paper (high = {1,3,7,8,9}, low = {2,4,5,6})
- confusion matrix shows tongue and feet as most confused — FBCSP's known weakness, see paper § 5.3

If any of those fail, the bug is upstream of the model: most likely the `epos` cue offset or the eval-label mapping. Re-check `understanding.txt` § 5.

If the checklist passes, we move on to the supervised neural baselines (EEGNet → ATCNet → Mamba). Each takes 5–20 min per subject on GPU; we'll add those experiment cells once these FBCSP numbers check out.

---
## 6. Experiment 2 — EEGNet (supervised, no augmentation)

Smallest deep CNN purpose-built for EEG (~2k params). See `understanding.txt § 8` for why it's experiment #2.

What this cell asks:
- **Does deep learning help at all over FBCSP?** (mean accuracy comparison)
- **Does the gain favour low performers?** (per-group Δ vs FBCSP — first test of the BCI-inefficiency hypothesis)

Runs `normal_results.EEG_results(aug_bool=False)` over all 9 subjects (300 epochs each, GPU). Result is pickled to `RESULTS_DIR/eegnet_noaug.pkl`. Set `FORCE_RERUN_EEGNET = True` to ignore the cache.

In [18]:
# Flip to True to ignore the pickle cache and retrain EEGNet.
FORCE_RERUN_EEGNET = False

eegnet_cache = os.path.join(RESULTS_DIR, 'eegnet_noaug.pkl')

# Same cache pattern as FBCSP — schema (acc, cm, runtime), backwards-compat for older pickles.
if os.path.exists(eegnet_cache) and not FORCE_RERUN_EEGNET:
    with open(eegnet_cache, 'rb') as f:
        loaded = pickle.load(f)
    eegnet_acc, eegnet_cm = loaded[0], loaded[1]
    eegnet_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {eegnet_runtime/60:.1f} min' if eegnet_runtime else 'prior runtime not recorded'
    print(f'Loaded cached EEGNet results ({msg})')
else:
    print('Training EEGNet on all 9 subjects (300 epochs each)...')
    from normal_results import EEG_results
    t0 = time.perf_counter()
    # aug_bool=False -> use raw training data, no S&R augmentation (that's experiment 5).
    eegnet_acc, eegnet_cm = EEG_results(aug_bool=False)
    eegnet_runtime = time.perf_counter() - t0
    with open(eegnet_cache, 'wb') as f:
        pickle.dump((eegnet_acc, eegnet_cm, eegnet_runtime), f)
    print(f'Wall time: {eegnet_runtime/60:.1f} min   cached -> {eegnet_cache}')

eegnet_acc = np.array(eegnet_acc)

# Per-subject view, alongside FBCSP for direct comparison.
print('\nPer-subject accuracy (EEGNet vs FBCSP):')
for i, (a_eeg, a_fb) in enumerate(zip(eegnet_acc, fbcsp_acc), 1):
    delta = a_eeg - a_fb
    print(f'  S{i:02d}: EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   Δ {delta:+.4f}')
print(f'  mean: EEGNet {eegnet_acc.mean():.4f}   FBCSP {fbcsp_acc.mean():.4f}   Δ {eegnet_acc.mean()-fbcsp_acc.mean():+.4f}')

# The headline test: does the gain over FBCSP favour LOW performers?
# Use the FBCSP-defined groups (never recompute from EEGNet).
high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]
gain_high = (eegnet_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (eegnet_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high performers {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  performers {[2,4,5,6]}  : {gain_low:+.4f}')
verdict = 'LOW benefit MORE — supports BCI-inefficiency hypothesis' if gain_low > gain_high else 'HIGH benefit more — counter to hypothesis'
print(f'  -> low minus high: {gain_low - gain_high:+.4f}   ({verdict})')

# Confusion matrix — expect tongue/feet less confused than under FBCSP.
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], eegnet_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached EEGNet results (prior run took 2.0 min)

Per-subject accuracy (EEGNet vs FBCSP):
  S01: EEGNet 0.8160   FBCSP 0.6944   Δ +0.1215
  S02: EEGNet 0.5556   FBCSP 0.5347   Δ +0.0208
  S03: EEGNet 0.8750   FBCSP 0.7986   Δ +0.0764
  S04: EEGNet 0.6215   FBCSP 0.5868   Δ +0.0347
  S05: EEGNet 0.6493   FBCSP 0.5208   Δ +0.1285
  S06: EEGNet 0.5625   FBCSP 0.4062   Δ +0.1562
  S07: EEGNet 0.6979   FBCSP 0.7847   Δ -0.0868
  S08: EEGNet 0.8056   FBCSP 0.7118   Δ +0.0938
  S09: EEGNet 0.7778   FBCSP 0.6944   Δ +0.0833
  mean: EEGNet 0.7068   FBCSP 0.6370   Δ +0.0698

Mean Δ over FBCSP, by group:
  high performers [1, 3, 7, 8, 9]: +0.0576
  low  performers [2, 4, 5, 6]  : +0.0851
  -> low minus high: +0.0274   (LOW benefit MORE — supports BCI-inefficiency hypothesis)

Confusion matrix (rows=true, cols=pred):
              Left   Right  Feet   Tongue
  true Left       455     69     71     53
  true Right       61    461     75     51
  true Feet        41     54    470     83
  true 

---
## 7. Experiment 3 — ATCNet (supervised, no augmentation)

ATCNet (Altaheri et al. 2022) = EEGNet front-end + sliding-window encoder + multi-head self-attention + temporal conv ("ATC" block), window predictions averaged. ~150k params. See `understanding.txt § 9` for the full motivation.

What this cell asks:
- **Does attention + longer temporal context help on top of EEGNet?**
- **Does the gain over FBCSP grow more for low performers than for high?**

Runs `normal_results.ATCNet_results(aug_bool=False)` over all 9 subjects (300 epochs each, GPU). Cached at `RESULTS_DIR/atcnet_noaug.pkl`. Set `FORCE_RERUN_ATCNET = True` to recompute.

In [19]:
# Flip to True to ignore the pickle cache and retrain ATCNet.
FORCE_RERUN_ATCNET = False

atcnet_cache = os.path.join(RESULTS_DIR, 'atcnet_noaug.pkl')

# Same cache pattern as EEGNet — schema (acc, cm, runtime).
if os.path.exists(atcnet_cache) and not FORCE_RERUN_ATCNET:
    with open(atcnet_cache, 'rb') as f:
        loaded = pickle.load(f)
    atcnet_acc, atcnet_cm = loaded[0], loaded[1]
    atcnet_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {atcnet_runtime/60:.1f} min' if atcnet_runtime else 'prior runtime not recorded'
    print(f'Loaded cached ATCNet results ({msg})')
else:
    print('Training ATCNet on all 9 subjects (300 epochs each)...')
    from normal_results import ATCNet_results
    t0 = time.perf_counter()
    # aug_bool=False -> raw training data only; S&R augmentation is experiment 5.
    atcnet_acc, atcnet_cm = ATCNet_results(aug_bool=False)
    atcnet_runtime = time.perf_counter() - t0
    with open(atcnet_cache, 'wb') as f:
        pickle.dump((atcnet_acc, atcnet_cm, atcnet_runtime), f)
    print(f'Wall time: {atcnet_runtime/60:.1f} min   cached -> {atcnet_cache}')

atcnet_acc = np.array(atcnet_acc)

# Per-subject view, alongside FBCSP and EEGNet — three columns of context.
print('\nPer-subject accuracy (ATCNet vs EEGNet vs FBCSP):')
for i, (a_atc, a_eeg, a_fb) in enumerate(zip(atcnet_acc, eegnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: ATCNet {a_atc:.4f}   EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_atc-a_fb:+.4f}   Δ_EEGNet {a_atc-a_eeg:+.4f}')
print(f'  mean: ATCNet {atcnet_acc.mean():.4f}   EEGNet {eegnet_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

# Headline: does ATCNet's gain over FBCSP favour low performers?
high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]
gain_high = (atcnet_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (atcnet_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high performers {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  performers {[2,4,5,6]}  : {gain_low:+.4f}')
verdict = 'LOW benefit MORE — supports BCI-inefficiency hypothesis' if gain_low > gain_high else 'HIGH benefit more — counter to hypothesis'
print(f'  -> low minus high: {gain_low - gain_high:+.4f}   ({verdict})')

print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], atcnet_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Training ATCNet on all 9 subjects (300 epochs each)...


/usr/local/lib/python3.12/dist-packages/braindecode/models/atcnet.py:284: UserWarning: n_times (512) is smaller than the minimum required (616) for the current model parameters configuration. Adjusting parameters to ensure compatibility.Reducing the kernel, pooling, and stride sizes accordingly.Scaling factor: 0.83
  warn(
/usr/local/lib/python3.12/dist-packages/braindecode/models/atcnet.py:284: UserWarning: n_times (512) is smaller than the minimum required (616) for the current model parameters configuration. Adjusting parameters to ensure compatibility.Reducing the kernel, pooling, and stride sizes accordingly.Scaling factor: 0.83
  warn(
/usr/local/lib/python3.12/dist-packages/braindecode/models/atcnet.py:284: UserWarning: n_times (512) is smaller than the minimum required (616) for the current model parameters configuration. Adjusting parameters to ensure compatibility.Reducing the kernel, pooling, and stride sizes accordingly.Scaling factor: 0.83
  warn(
/usr/local/lib/python3.12

Wall time: 10.6 min   cached -> /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project/results/atcnet_noaug.pkl

Per-subject accuracy (ATCNet vs EEGNet vs FBCSP):
  S01: ATCNet 0.7188   EEGNet 0.8160   FBCSP 0.6944   Δ_FBCSP +0.0243   Δ_EEGNet -0.0972
  S02: ATCNet 0.5417   EEGNet 0.5556   FBCSP 0.5347   Δ_FBCSP +0.0069   Δ_EEGNet -0.0139
  S03: ATCNet 0.8438   EEGNet 0.8750   FBCSP 0.7986   Δ_FBCSP +0.0451   Δ_EEGNet -0.0312
  S04: ATCNet 0.6493   EEGNet 0.6215   FBCSP 0.5868   Δ_FBCSP +0.0625   Δ_EEGNet +0.0278
  S05: ATCNet 0.6806   EEGNet 0.6493   FBCSP 0.5208   Δ_FBCSP +0.1597   Δ_EEGNet +0.0312
  S06: ATCNet 0.5868   EEGNet 0.5625   FBCSP 0.4062   Δ_FBCSP +0.1806   Δ_EEGNet +0.0243
  S07: ATCNet 0.6736   EEGNet 0.6979   FBCSP 0.7847   Δ_FBCSP -0.1111   Δ_EEGNet -0.0243
  S08: ATCNet 0.7500   EEGNet 0.8056   FBCSP 0.7118   Δ_FBCSP +0.0382   Δ_EEGNet -0.0556
  S09: ATCNet 0.7465   EEGNet 0.7778   FBCSP 0.6944   Δ_FBCSP +0.0521   Δ_EEGNet -0.0312
  mean: ATCNet 0.6879   E

---
## 8. Experiment 4 — MI-Mamba (supervised, no augmentation)

MI-Mamba wraps the Mamba selective state-space backbone (Gu & Dao 2023) for motor-imagery EEG. State-space models give long-range context with O(N) cost instead of attention's O(N²). See `understanding.txt § 10`.

Note: trained for **500 epochs** (vs 300 for EEGNet/ATCNet) — that's the value `normal_results.Mamba_results` was written with, kept for fidelity.

What this cell asks:
- **Does a modern non-attention sequence model beat ATCNet?**
- **Same group-Δ test** — does Mamba's gain over FBCSP favour low performers?

Runs `normal_results.Mamba_results(aug_bool=False)`. Cached at `RESULTS_DIR/mamba_noaug.pkl`. Set `FORCE_RERUN_MAMBA = True` to recompute.

In [20]:
# Flip to True to ignore the pickle cache and retrain Mamba.
FORCE_RERUN_MAMBA = False

mamba_cache = os.path.join(RESULTS_DIR, 'mamba_noaug.pkl')

# Same cache pattern as the other supervised models.
if os.path.exists(mamba_cache) and not FORCE_RERUN_MAMBA:
    with open(mamba_cache, 'rb') as f:
        loaded = pickle.load(f)
    mamba_acc, mamba_cm = loaded[0], loaded[1]
    mamba_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {mamba_runtime/60:.1f} min' if mamba_runtime else 'prior runtime not recorded'
    print(f'Loaded cached Mamba results ({msg})')
else:
    print('Training MI-Mamba on all 9 subjects (500 epochs each)...')
    from normal_results import Mamba_results
    t0 = time.perf_counter()
    # aug_bool=False -> raw training data only; S&R augmentation is experiment 5.
    mamba_acc, mamba_cm = Mamba_results(aug_bool=False)
    mamba_runtime = time.perf_counter() - t0
    with open(mamba_cache, 'wb') as f:
        pickle.dump((mamba_acc, mamba_cm, mamba_runtime), f)
    print(f'Wall time: {mamba_runtime/60:.1f} min   cached -> {mamba_cache}')

mamba_acc = np.array(mamba_acc)

# Per-subject view across all four supervised models.
print('\nPer-subject accuracy (Mamba vs ATCNet vs EEGNet vs FBCSP):')
for i, (a_m, a_atc, a_eeg, a_fb) in enumerate(zip(mamba_acc, atcnet_acc, eegnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: Mamba {a_m:.4f}   ATCNet {a_atc:.4f}   EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_m-a_fb:+.4f}   Δ_ATCNet {a_m-a_atc:+.4f}')
print(f'  mean: Mamba {mamba_acc.mean():.4f}   ATCNet {atcnet_acc.mean():.4f}   '
      f'EEGNet {eegnet_acc.mean():.4f}   FBCSP {fbcsp_acc.mean():.4f}')

# Headline: does Mamba's gain over FBCSP favour low performers?
high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]
gain_high = (mamba_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (mamba_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high performers {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  performers {[2,4,5,6]}  : {gain_low:+.4f}')
verdict = 'LOW benefit MORE — supports BCI-inefficiency hypothesis' if gain_low > gain_high else 'HIGH benefit more — counter to hypothesis'
print(f'  -> low minus high: {gain_low - gain_high:+.4f}   ({verdict})')

print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], mamba_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Training MI-Mamba on all 9 subjects (500 epochs each)...
Wall time: 3.3 min   cached -> /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project/results/mamba_noaug.pkl

Per-subject accuracy (Mamba vs ATCNet vs EEGNet vs FBCSP):
  S01: Mamba 0.6007   ATCNet 0.7188   EEGNet 0.8160   FBCSP 0.6944   Δ_FBCSP -0.0938   Δ_ATCNet -0.1181
  S02: Mamba 0.3611   ATCNet 0.5417   EEGNet 0.5556   FBCSP 0.5347   Δ_FBCSP -0.1736   Δ_ATCNet -0.1806
  S03: Mamba 0.7326   ATCNet 0.8438   EEGNet 0.8750   FBCSP 0.7986   Δ_FBCSP -0.0660   Δ_ATCNet -0.1111
  S04: Mamba 0.4375   ATCNet 0.6493   EEGNet 0.6215   FBCSP 0.5868   Δ_FBCSP -0.1493   Δ_ATCNet -0.2118
  S05: Mamba 0.2847   ATCNet 0.6806   EEGNet 0.6493   FBCSP 0.5208   Δ_FBCSP -0.2361   Δ_ATCNet -0.3958
  S06: Mamba 0.3472   ATCNet 0.5868   EEGNet 0.5625   FBCSP 0.4062   Δ_FBCSP -0.0590   Δ_ATCNet -0.2396
  S07: Mamba 0.5069   ATCNet 0.6736   EEGNet 0.6979   FBCSP 0.7847   Δ_FBCSP -0.2778   Δ_ATCNet -0.1667
  S08: Mamba 0.6736   ATCNet 0.750